In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_48.pickle

In [ ]:
%%PandasProfile
### cell 48 ###

# Optimized code

def count_then_return_percent_60(dataframe, x_axis_title):
    # compute raw counts including NaN, then normalize by non-null count
    s = dataframe[x_axis_title]
    counts = s.value_counts(dropna=False)
    # use counts.sum() excluding NaN if we want to match original denominator: s.count() == non-null count
    denom = s.count()
    return (counts.mul(100.0 / denom)
                  .round(1))


def combine_subset_of_data_from_multiple_years_60(question_of_interest, x_axis_title, include_2017=False):
    # mapping of year to its dataframe
    dfs = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10,
    }
    if include_2017:
        dfs['2017'] = responses_df_2017

    # build a list of (year, df) tuples, sorted by year ascending for final output
    items = []
    for year, df in dfs.items():
        # compute percentages series, convert to DataFrame, add columns
        pct = count_then_return_percent_60(df, question_of_interest).sort_index()
        df_temp = pct.to_frame(name='percentage')
        df_temp['year'] = year
        df_temp[x_axis_title] = df_temp.index
        items.append((year, df_temp))

    # order the dataframes by year
    sorted_items = sorted(items, key=lambda x: x[0])
    # extract dfs in order
    df_list = [t[1] for t in sorted_items]

    # concatenate once
    df_combined = pd.concat(df_list, ignore_index=True)
    # ensure column order: percentage, year, x_axis_title
    cols = ['percentage', 'year', x_axis_title]
    return df_combined[cols]

# Usage
question_of_interest_cell60 = 'Does your current employer incorporate machine learning methods into their business?'
title_for_x_axis_cell60 = ''
maturity_df_combined = (
    combine_subset_of_data_from_multiple_years_60(
        question_of_interest_cell60,
        title_for_x_axis_cell60
    )
    .sort_values(by=['year', 'percentage'], ascending=True)
)
maturity_df_combined.info()